In [12]:
import numpy as np
import pandas as pd

df_nea = pd.read_csv('../data/near_earth_asteroids_2025 (1).csv')
df_approaches = pd.read_csv('../data/asteroid_close_approaches_2015_2035 (1).csv')

aggr_approaches = df_approaches.groupby('designation').agg(
    min_dist_au=('distance_au', 'min'),
    max_vel_kms=('velocity_km_s', 'max'),
    approach_count=('designation', 'count'), #how many times it passes
).reset_index()

df_master = pd.merge(
    df_nea,
    aggr_approaches,
    left_on='pdes',
    right_on='designation',
    how='left'
)

print(f"Master Table Shape: {df_master.shape}")
df_master.head()

# force these cols as strings
df_master['pdes'] = df_master['pdes'].astype(str)
df_master['condition_code'] = df_master['condition_code'].astype(str)

cols_to_drop = [
    'pdes', 'name', 'spkid', 'full_name', 'designation',
    'first_obs', 'last_obs', 'data_arc', 'data_arc_years',
    'albedo', 'rot_per'
]

df_cleaned = df_master.drop(columns=[c for c in cols_to_drop if c in df_master.columns])

df_cleaned['min_dist_au'] = df_cleaned['min_dist_au'].fillna(2.0)
df_cleaned['max_vel_kms'] = df_cleaned['max_vel_kms'].fillna(0.0)
df_cleaned['approach_count'] = df_cleaned['approach_count'].fillna(0)

print(f"Features remaining for training: {df_cleaned.shape[1]}")

df_cleaned['pha'] = df_cleaned['pha'].map({True: 1, False: 0, 'True': 1, 'False': 0})

df_cleaned = df_cleaned.fillna(df_cleaned.median(numeric_only=True))

df_cleaned.head()

C:\Users\ghita\AppData\Local\Temp\ipykernel_22256\2888791172.py:4: DtypeWarning: Columns (0: name) have mixed types. Specify dtype option on import or set low_memory=False.
  df_nea = pd.read_csv('../data/near_earth_asteroids_2025 (1).csv')


Master Table Shape: (41281, 33)
Features remaining for training: 22


,pha,H,diameter_km,diameter_m,diameter_is_estimated,size_category,class,e,a,i,...,per,per_y,moid_au,moid_km,moid_lunar_distances,n,condition_code,min_dist_au,max_vel_kms,approach_count
0,0,10.39,16.84000,16840.0,False,Large (>1 km) — City killer+,AMO,0.2228,1.458,10.83,...,643.0,1.76,0.1480,22140485.0,57.60,0.5598,0.0,2.000000,0.000000,0.0
1,0,15.59,2.70683,2706.8,True,Large (>1 km) — City killer+,AMO,0.5466,2.637,11.57,...,1560.0,4.28,0.2010,30069172.0,78.22,0.2302,0.0,2.000000,0.000000,0.0
2,0,13.82,4.20000,4200.0,False,Large (>1 km) — City killer+,AMO,0.5712,2.474,9.40,...,1420.0,3.89,0.0797,11922950.0,31.02,0.2533,0.0,0.082198,8.248699,1.0
3,0,9.17,37.67500,37675.0,False,Large (>1 km) — City killer+,AMO,0.5332,2.665,26.68,...,1590.0,4.35,0.3430,51312070.0,133.49,0.2266,0.0,2.000000,0.000000,0.0
4,0,17.37,1.00000,1000.0,False,Large (>1 km) — City killer+,AMO,0.4346,1.920,11.87,...,972.0,2.66,0.1080,16156570.0,42.03,0.3705,0.0,2.000000,0.000000,0.0


In [13]:
import pandas as pd
import numpy as np
def map_size_text(row):
    text = str(row['size_category'])
    h_val = row['H']
    if 'City Killer' in text: return 4
    if 'Regional damage' in text: return 3
    if 'Local damage' in text: return 2
    if 'Small' in text: return 1
    if 'Tiny' in text: return 0

    if h_val < 18: return 3
    if h_val > 25: return 0
    return 1

df_cleaned['size_category'] = df_cleaned.apply(map_size_text, axis=1)
df_final = pd.get_dummies(df_cleaned, columns=['class'], prefix='orbit')

df_final['condition_code'] = pd.to_numeric(df_final['condition_code'], errors='coerce').fillna(0).astype(int)

text_left = df_final.select_dtypes(include=['object']).columns.tolist()
if not text_left:
    print("100% numeric.")
else:
    print(f"still text: {text_left}")

print("\n--- Correlation with PHA (Hazard Status) ---")
df_final = df_final.astype(float)
print(df_final.corr()['pha'].sort_values(ascending=False))

df_final.head()

100% numeric.

--- Correlation with PHA (Hazard Status) ---
pha                      1.000000
size_category            0.275090
diameter_m               0.157567
diameter_km              0.157566
e                        0.154118
orbit_APO                0.148545
i                        0.063835
min_dist_au              0.047538
max_vel_kms              0.042759
orbit_IEO                0.018827
ad                       0.016443
a                        0.007393
per                     -0.000653
per_y                   -0.000654
orbit_ATE               -0.002992
n                       -0.018186
approach_count          -0.035956
orbit_AMO               -0.153816
diameter_is_estimated   -0.154616
moid_au                 -0.155231
moid_km                 -0.155231
moid_lunar_distances    -0.155231
q                       -0.171601
H                       -0.307289
condition_code          -0.312747
Name: pha, dtype: float64


,pha,H,diameter_km,diameter_m,diameter_is_estimated,size_category,e,a,i,q,...,moid_lunar_distances,n,condition_code,min_dist_au,max_vel_kms,approach_count,orbit_AMO,orbit_APO,orbit_ATE,orbit_IEO
0,0.0,10.39,16.84000,16840.0,0.0,3.0,0.2228,1.458,10.83,1.133,...,57.60,0.5598,0.0,2.000000,0.000000,0.0,1.0,0.0,0.0,0.0
1,0.0,15.59,2.70683,2706.8,1.0,3.0,0.5466,2.637,11.57,1.195,...,78.22,0.2302,0.0,2.000000,0.000000,0.0,1.0,0.0,0.0,0.0
2,0.0,13.82,4.20000,4200.0,0.0,3.0,0.5712,2.474,9.40,1.061,...,31.02,0.2533,0.0,0.082198,8.248699,1.0,1.0,0.0,0.0,0.0
3,0.0,9.17,37.67500,37675.0,0.0,3.0,0.5332,2.665,26.68,1.244,...,133.49,0.2266,0.0,2.000000,0.000000,0.0,1.0,0.0,0.0,0.0
4,0.0,17.37,1.00000,1000.0,0.0,3.0,0.4346,1.920,11.87,1.085,...,42.03,0.3705,0.0,2.000000,0.000000,0.0,1.0,0.0,0.0,0.0
